# F_spont_choice_lick_pupil.ipynb

## Purpose
Analyzes whether pupil diameter responds differently to spontaneous versus task-related licks. Generates figure panels comparing pupil traces aligned to each lick type across Hopkins sessions (which have pupil tracking). Analysis is session-level only — no spatial coordinates are available for these sessions.

## Prerequisites
- `code/data_management/hopkins_session_assets.csv` — session list (only sessions with pupil data)
- Per-session NWB and lick timing files via `load_licks()` / `load_licks_video()`
- Per-session pupil data via `load_pupil()` (requires `.mat` pupil file and DLC CSV in sorted_dir_curated)
- `combined_session_tbl/combined_beh_sessions.pkl` — for trial timing

## Environment Setup and Imports

In [ ]:
import os
import pathlib
import sys

try:
    _here = pathlib.Path(__file__).resolve().parent
except NameError:
    try:
        _here = pathlib.Path(__vsc_ipynb_file__).resolve().parent
    except NameError:
        _here = pathlib.Path(os.getcwd())

_beh_ephys_root = str((_here / '../..').resolve())
if _beh_ephys_root not in sys.path:
    sys.path.insert(0, _beh_ephys_root)

import json
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import pearsonr

from utils.beh_functions import get_session_tbl, session_dirs
from utils.capsule_migration import CAPSULE_ROOT, capsule_directories
from utils.lick_utils import load_licks, load_licks_video
from utils.panel_utils import columns_to_df, save_panel_csv, save_panels
from utils.photometry_utils import bin_timeseries_around_align
from utils.pupil_utils import load_pupil

warnings.filterwarnings('ignore')
capsule_dirs = capsule_directories()

%matplotlib inline

## Analysis Configuration

In [ ]:
target_folder = os.path.join(capsule_dirs["manuscript_fig_dir"], 'F_lick_pupil_compare')
os.makedirs(target_folder, exist_ok=True)

## Load Sessions

Only Hopkins sessions have pupil tracking. Load from `hopkins_session_assets.csv`.

In [ ]:
data_df = pd.read_csv(CAPSULE_ROOT + '/code/data_management/hopkins_session_assets.csv')
sessions = data_df['session_id'].dropna().astype(str).tolist()
print(f'Total sessions: {len(sessions)}')

## Define Pupil Lick PSTH Computation Function

Computes pupil diameter traces aligned to lick onset for a given session. Distinguishes between:
- **In-trial licks**: licks occurring 0–1.5 s after go cue
- **Out-of-trial licks (behavior)**: spontaneous licks detected by behavioral sensors
- **Out-of-trial licks (video)**: spontaneous licks detected only in video

Pupil is z-scored within session before alignment.

In [ ]:
def get_pupil_lick_psth(session):
    bin_size = 0.5
    step_size = 0.1
    tb = -5.0
    tf = 5.0
    in_trial_period = (0, 1.5)
    min_licks = 5

    pupil_data = load_pupil(session)
    if pupil_data is None:
        return None

    pupil_times = pupil_data['pupil_times']
    pupil_z = pupil_data['pupil_diameter_z']

    licks = load_licks_video(session)
    licks_beh = load_licks(session)
    video_loaded = licks is not None
    if not video_loaded:
        licks = licks_beh

    session_tbl = get_session_tbl(session)

    lick_starts_L = licks['lick_trains_L']['train_starts']
    lick_starts_R = licks['lick_trains_R']['train_starts']
    lick_starts_all = np.concatenate([lick_starts_L, lick_starts_R])

    # restrict to pupil recording window
    t_min = np.nanmin(pupil_times)
    t_max = np.nanmax(pupil_times)
    lick_mask = (lick_starts_all >= t_min) & (lick_starts_all <= t_max)
    lick_starts_all = lick_starts_all[lick_mask]

    # label in-trial vs out-of-trial
    in_trial_mask = np.zeros(len(lick_starts_all), dtype=bool)
    for _, trial in session_tbl.iterrows():
        trial_start = trial['goCue_start_time']
        in_trial_mask |= (
            (lick_starts_all >= trial_start + in_trial_period[0]) &
            (lick_starts_all <= trial_start + in_trial_period[1])
        )

    lick_starts_in_trial = lick_starts_all[in_trial_mask]
    lick_starts_out_trial = lick_starts_all[~in_trial_mask]

    ts = {'time': pupil_times, 'value': pupil_z}

    def _align(align_times):
        if len(align_times) < min_licks:
            return np.full(int((tf - tb - bin_size) / step_size), np.nan)
        mat, bin_centers = bin_timeseries_around_align(
            ts, align_times, step_size=step_size,
            bin_size=bin_size, t_start=tb, t_stop=tf
        )
        return np.nanmean(mat, axis=0), bin_centers

    result_in = _align(lick_starts_in_trial)
    result_out = _align(lick_starts_out_trial)

    if isinstance(result_in, tuple):
        psth_in, bin_centers = result_in
    else:
        psth_in = result_in
        bin_centers = None

    if isinstance(result_out, tuple):
        psth_out_beh, _ = result_out
    else:
        psth_out_beh = result_out

    # separate video-only licks from behavior licks for out-of-trial
    if video_loaded:
        lick_starts_beh_all = np.concatenate([
            licks_beh['lick_trains_L']['train_starts'],
            licks_beh['lick_trains_R']['train_starts']
        ])
        close_beh_mask = np.zeros(len(lick_starts_all), dtype=bool)
        for i, lick in enumerate(lick_starts_all):
            if len(lick_starts_beh_all) == 0:
                break
            if np.min(np.abs(lick_starts_beh_all - lick)) <= 0.2:
                close_beh_mask[i] = True
        lick_starts_out_video = lick_starts_all[~in_trial_mask & ~close_beh_mask]
        result_out_video = _align(lick_starts_out_video)
        psth_out_video = result_out_video[0] if isinstance(result_out_video, tuple) else result_out_video
    else:
        psth_out_video = np.full_like(psth_in, np.nan)

    return {
        'session': session,
        'psth_in': psth_in,
        'psth_out_beh': psth_out_beh,
        'psth_out_video': psth_out_video,
        'bin_centers': bin_centers,
        'n_in': int(in_trial_mask.sum()),
        'n_out': int((~in_trial_mask).sum()),
        'error_prop': pupil_data.get('error_prop', np.nan),
    }

## Compute Pupil-Aligned Traces for All Sessions

In [ ]:
results_raw = Parallel(n_jobs=-1)(delayed(get_pupil_lick_psth)(s) for s in sessions)
results_raw = [r for r in results_raw if r is not None]
results_df = pd.DataFrame(results_raw)
print(f'Sessions with pupil data: {len(results_df)}')

## Organize Results and Baseline-Subtract

Baseline is computed from the −5 to −1 s pre-lick period of the in-trial trace.

In [ ]:
bin_centers = results_df['bin_centers'].dropna().iloc[0]

psth_in = np.array(results_df['psth_in'].tolist())
psth_out_beh = np.array(results_df['psth_out_beh'].tolist())
psth_out_video = np.array(results_df['psth_out_video'].tolist())

bl_mask = (bin_centers >= -5) & (bin_centers < -1)
bl = np.nanmean(psth_in[:, bl_mask], axis=1, keepdims=True)

psth_in = psth_in - bl
psth_out_beh = psth_out_beh - bl
psth_out_video = psth_out_video - bl

ani_ids = [s.split('_')[1] for s in results_df['session'].values]

## Heatmap Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
tb, tf = bin_centers[0], bin_centers[-1]

for ax, psth, title in zip(
    axes,
    [psth_in, psth_out_beh, psth_out_video],
    ['Pupil: In-Trial Licks', 'Pupil: Out-of-Trial Licks (Beh)', 'Pupil: Out-of-Trial Licks (Video only)']
):
    im = ax.imshow(
        psth, aspect='auto',
        extent=[tb, tf, 0, psth.shape[0]],
        origin='lower', cmap='RdBu_r'
    )
    ax.set_title(title)
    ax.set_xlabel('Time from lick (s)')
    ax.axvline(0, color='w', linestyle='--', linewidth=0.8)
    # draw line between animals
    ani_arr = np.array(ani_ids)
    for ani_id in np.unique(ani_arr):
        idx = np.where(ani_arr == ani_id)[0]
        ax.axhline(idx[0], color='w', linestyle='--', linewidth=0.5)
    plt.colorbar(im, ax=ax, label='Pupil diameter (z, bl-sub)')

axes[0].set_ylabel('Session')
plt.tight_layout()
save_panels(fig, target_folder, 'pupil_lick_heatmap', [], exts=('pdf', 'svg'))

## Mean Traces

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

def _plot_mean_sem(ax, t, mat, label, color):
    valid = ~np.all(np.isnan(mat), axis=1)
    m = np.nanmean(mat[valid], axis=0)
    sem = np.nanstd(mat[valid], axis=0) / np.sqrt(valid.sum())
    ax.plot(t, m, color=color, label=label)
    ax.fill_between(t, m - sem, m + sem, color=color, alpha=0.25)

_plot_mean_sem(ax, bin_centers, psth_in, 'In-trial licks', 'tab:blue')
_plot_mean_sem(ax, bin_centers, psth_out_beh, 'Out-of-trial licks (beh)', 'tab:orange')

ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
ax.set_xlabel('Time from lick (s)')
ax.set_ylabel('Pupil diameter (z, bl-sub)')
ax.legend()
ax.set_title('Mean ± SEM pupil response aligned to lick onset')
plt.tight_layout()
save_panels(fig, target_folder, 'pupil_lick_mean_trace', [], exts=('pdf', 'svg'))
# panel CSV
save_panel_csv(
    columns_to_df(**{
        'time': bin_centers,
        'pupil_in_mean': np.nanmean(psth_in, axis=0),
        'pupil_out_mean': np.nanmean(psth_out_beh, axis=0),
    }),
    target_folder, 'pupil_lick_mean_trace', []
)

## Extract Peak Response Metrics

For each session, extract the peak pupil response in the 0–2 s peri-lick window for in-trial and out-of-trial licks.

In [ ]:
def cal_peak(psth_mat, time_vector, search_window):
    window_mask = (time_vector >= search_window[0]) & (time_vector <= search_window[1])
    peak_vals = []
    for row in psth_mat:
        window = row[window_mask]
        peak_vals.append(np.nanmax(window) if np.any(np.isfinite(window)) else np.nan)
    return np.array(peak_vals)

search_window = (0, 2.0)
peak_val_in = cal_peak(psth_in, bin_centers, search_window)
peak_val_out = cal_peak(psth_out_beh, bin_centers, search_window)

results_df['peak_val_in'] = peak_val_in
results_df['peak_val_out'] = peak_val_out

## Scatter: In-Trial vs Out-of-Trial Peak Pupil Response

In [ ]:
x_val = 'peak_val_in'
y_val = 'peak_val_out'
color_code = 'peak_val_in'

sort_ind = np.argsort(results_df[color_code].values)

fig = plt.figure(figsize=(6, 6))
gs = fig.add_gridspec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4], wspace=0.05, hspace=0.05)
ax = fig.add_subplot(gs[1, 0])
ax_histx = fig.add_subplot(gs[0, 0], sharex=ax)
ax_histy = fig.add_subplot(gs[1, 1], sharey=ax)
ax_histx.tick_params(axis='x', labelbottom=False)
ax_histy.tick_params(axis='y', labelleft=False)

g = ax.scatter(
    results_df[x_val][sort_ind], results_df[y_val][sort_ind],
    alpha=1, edgecolors='w',
    c=results_df[color_code][sort_ind], cmap='viridis', s=60,
    vmin=np.nanquantile(results_df[color_code].values, 0.05),
    vmax=np.nanquantile(results_df[color_code].values, 0.95)
)

_xlim = [results_df[x_val].min(), results_df[x_val].max()]
ax.plot(_xlim, _xlim, 'k--')

ax_histx.hist(results_df[x_val].dropna(), bins=20, color='gray', edgecolor='none')
ax_histy.hist(results_df[y_val].dropna(), bins=20, orientation='horizontal', color='gray', edgecolor='none')

cbar_ax = fig.add_subplot(gs[0, 1])
plt.colorbar(g, cax=cbar_ax, label=color_code)

filt = ~np.isnan(results_df[x_val]) & ~np.isnan(results_df[y_val])
if filt.sum() > 2:
    corr_xy = pearsonr(results_df[x_val][filt], results_df[y_val][filt])
    _xv = results_df[x_val][filt].values
    _yv = results_df[y_val][filt].values
    _slope, _intercept = np.polyfit(_xv, _yv, 1)
    _xfit = np.linspace(_xv.min(), _xv.max(), 100)
    _rng = np.random.default_rng(0)
    _n_boot = 1000
    _boot_preds = np.empty((_n_boot, _xfit.size))
    for _b in range(_n_boot):
        _idx = _rng.integers(0, len(_xv), size=len(_xv))
        _s, _i = np.polyfit(_xv[_idx], _yv[_idx], 1)
        _boot_preds[_b] = _s * _xfit + _i
    ax.fill_between(_xfit, np.percentile(_boot_preds, 2.5, axis=0),
                    np.percentile(_boot_preds, 97.5, axis=0),
                    color='red', alpha=0.2, edgecolor='none')
    ax.plot(_xfit, _slope * _xfit + _intercept, color='red', linewidth=1.5)
    ax_histx.set_title(f'{y_val} vs {x_val} (r={corr_xy[0]:.2f}, p={corr_xy[1]:.3e})')

ax.set_xlabel('Peak pupil response: in-trial licks (z)')
ax.set_ylabel('Peak pupil response: out-of-trial licks (z)')
plt.tight_layout()
save_panels(fig, target_folder, 'pupil_peak_in_vs_out', [], exts=('svg', 'pdf'), bbox_inches='tight', dpi=300)
save_panel_csv(
    columns_to_df(**{'peak_val_in': results_df['peak_val_in'].values,
                     'peak_val_out': results_df['peak_val_out'].values}),
    target_folder, 'pupil_peak_in_vs_out', []
)